In [1]:
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

/Users/rishaan/miniforge3/envs/langchain/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Read the PDF
loader = PyPDFDirectoryLoader('./us_census')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census/acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.\n1 Public policy changes included \nthe renewal of the Public Health Emergency, w

In [3]:
len(documents)

316

In [4]:
#HuggingFace Embeddings
embedding = HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-small-en-v1.5',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True}
)

/var/folders/qv/7r3vg1ns6qbgvs0t3_dxc3qm0000gn/T/ipykernel_2190/1604856201.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceBgeEmbeddings(


In [5]:
import numpy as np
embedd = np.array(embedding.embed_query(documents[0].page_content))
embedd

array([-0.07903483, -0.01134113, -0.023121  ,  0.02844467,  0.05053339,
        0.05317826, -0.01907787,  0.0345603 , -0.10211373, -0.02915701,
        0.08524261,  0.05650727, -0.02545442, -0.03308494, -0.00635735,
        0.04090862, -0.00628107,  0.0035674 , -0.03854132,  0.03667679,
       -0.04289804,  0.03425257, -0.03116897, -0.03793732,  0.0172839 ,
        0.01214924,  0.00653121,  0.01463568, -0.05529053, -0.15320703,
        0.00730846,  0.03202944, -0.04701128, -0.01595979,  0.01874445,
        0.02642935, -0.02306375,  0.08438037,  0.04182493,  0.05278177,
       -0.03057601,  0.01564265, -0.01689073,  0.00529408, -0.02417439,
        0.00412994, -0.01889933, -0.0015063 , -0.00836946, -0.03390065,
        0.03515958, -0.00553132,  0.04910939,  0.0597186 ,  0.05615968,
       -0.05105155,  0.01475136, -0.01849963, -0.03284639,  0.03576629,
        0.04947708, -0.00938879, -0.26202118,  0.09750336,  0.01715689,
        0.04781392, -0.0055632 , -0.00298304, -0.02207354, -0.04

In [6]:
embedd.shape

(384,)

In [7]:
vectors = FAISS.from_documents(documents, embedding)

In [8]:
#Query using Similarity Search
query="WHAT IS HEALTH INSURANCE COVERAGE?"
relevant_documents = vectors.similarity_search(query)
print(relevant_documents[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [9]:
retriever = vectors.as_retriever(search_type='similarity', search_kwargs={'k':3})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceBgeEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x16ebb1880>, search_kwargs={'k': 3})

In [ ]:
import os
huugingface_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

##### The Hugging Face Hub is an platform with over 350k models, 75k datasets, and 150k demo apps (Spaces), all open source and publicly available, in an online platform where people can easily collaborate and build ML together.

In [21]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)

In [22]:
# Create pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=250,
    temperature=0.1,
    do_sample=True,
    return_full_text=False
)

Device set to use mps:0


In [23]:
# Wrap pipeline in LangChain
llm = HuggingFacePipeline(pipeline=pipe)

In [24]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """

In [25]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])

In [27]:
retrieval = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={'prompt':prompt}
)

In [28]:
query="""DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022"""

In [29]:
# Call the QA chain with our query.
result = retrieval.invoke({'query': query})
print(result['result'])

1. Which state had the highest uninsured rate in 2022?
 2. Which state had the lowest uninsured rate in 2022?
 3. Which state had the highest uninsured rate in 2021?
 4. Which state had the lowest uninsured rate in 2021?
 5. Which state had the highest uninsured rate in 2022, but not in 2021?
 6. Which state had the lowest uninsured rate in 2022, but not in 2021?
 7. Which state had the highest uninsured rate in 2022, but not in 2021?
 8. Which state had the highest uninsured rate in 2021, but not in 2022?
 9. Which state had the highest uninsured rate in 2022, but not in 2021?
 10. Which state had the highest uninsured rate in 2022, but not in 20


In [30]:
result

{'query': 'DIFFERENCES IN THE\nUNINSURED RATE BY STATE\nIN 2022',
 'result': '1. Which state had the highest uninsured rate in 2022?\n 2. Which state had the lowest uninsured rate in 2022?\n 3. Which state had the highest uninsured rate in 2021?\n 4. Which state had the lowest uninsured rate in 2021?\n 5. Which state had the highest uninsured rate in 2022, but not in 2021?\n 6. Which state had the lowest uninsured rate in 2022, but not in 2021?\n 7. Which state had the highest uninsured rate in 2022, but not in 2021?\n 8. Which state had the highest uninsured rate in 2021, but not in 2022?\n 9. Which state had the highest uninsured rate in 2022, but not in 2021?\n 10. Which state had the highest uninsured rate in 2022, but not in 20',
 'source_documents': [Document(id='f5876bc9-afb9-4499-9b7e-8e7bb355c652', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 